# Phase 7 — Customer Intelligence Demo

This notebook demonstrates the Customer Intelligence Engine:
1. Trust score calculation and tier distribution
2. Sentiment trend analysis (6-month rolling window)
3. Risk profiling and segmentation
4. Product recommendations engine
5. Full customer 360 profile

**Run from the project root:** `jupyter notebook notebooks/03_customer_intelligence_demo.ipynb`

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

os.environ.setdefault('DATABASE_URL', 'sqlite:///../banking.db')
os.environ.setdefault('EMBEDDING_BACKEND', 'sentence_transformers')
os.environ.setdefault('VECTOR_DB', 'chroma')
os.environ.setdefault('CHROMA_PERSIST_DIR', '../data/chroma_db')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
from sqlalchemy import create_engine

engine = create_engine(os.environ['DATABASE_URL'])
customers = pd.read_sql('SELECT * FROM customers LIMIT 100', engine)
print(f'Loaded {len(customers)} sample customers')

## 1. Trust Score Distribution

In [ ]:
from src.models.trust_scorer import TrustScoreCalculator

calc = TrustScoreCalculator()
scores = []
for cid in customers['customer_id'].head(50):
    r = calc.calculate(cid)
    scores.append({'customer_id': cid, 'score': r['score'], 'tier': r['tier']})

df_scores = pd.DataFrame(scores)
print(f'Average trust score: {df_scores["score"].mean():.1f}')
print(df_scores['tier'].value_counts())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df_scores['score'].hist(ax=ax1, bins=20, color='steelblue', edgecolor='white')
ax1.axvline(40, color='red', linestyle='--', label='High Risk boundary')
ax1.axvline(70, color='green', linestyle='--', label='Trusted boundary')
ax1.set_xlabel('Trust Score'); ax1.set_ylabel('Count')
ax1.set_title('Trust Score Distribution'); ax1.legend()

tier_counts = df_scores['tier'].value_counts()
colors = {'Trusted': '#10b981', 'Moderate': '#f59e0b', 'High Risk': '#ef4444'}
tier_counts.plot(kind='pie', ax=ax2, autopct='%1.0f%%',
                 colors=[colors.get(t, '#64748b') for t in tier_counts.index])
ax2.set_title('Trust Tier Breakdown'); ax2.set_ylabel('')
plt.tight_layout()
plt.show()

## 2. Sentiment Trends

In [ ]:
df_interactions = pd.read_sql(
    "SELECT customer_id, sentiment_score, timestamp FROM interactions ORDER BY timestamp",
    engine
)
df_interactions['timestamp'] = pd.to_datetime(df_interactions['timestamp'])
df_interactions['month'] = df_interactions['timestamp'].dt.to_period('M')

monthly = df_interactions.groupby('month')['sentiment_score'].agg(['mean', 'count'])
monthly.index = monthly.index.astype(str)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
monthly['mean'].plot(ax=ax1, color='steelblue', marker='o', label='Avg Sentiment')
monthly['count'].plot(ax=ax2, color='orange', alpha=0.5, kind='bar', label='Interactions')
ax1.set_ylabel('Average Sentiment Score (-1 to +1)')
ax2.set_ylabel('Interaction Count')
ax1.set_title('Monthly Sentiment Trend (All Customers)')
ax1.axhline(0, color='gray', linestyle='--')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Product Recommendations

In [ ]:
from src.intelligence.recommender import ProductRecommender

recommender = ProductRecommender()
sample_id = customers.iloc[0]['customer_id']
recs = recommender.get_recommendations(sample_id)

print(f'Recommendations for {sample_id}:')
for i, rec in enumerate(recs, 1):
    print(f'  {i}. {rec.get("product", "?")} — {rec.get("reason", "?")} (score: {rec.get("score", 0):.2f})')

## 4. Full Customer 360 Profile

In [ ]:
from src.rag.customer_context import get_full_customer_context

ctx = get_full_customer_context(sample_id)

p = ctx['profile']
print('=== CUSTOMER 360 PROFILE ===')
print(f'Name:         {p.get("name")}')
print(f'Account Type: {p.get("account_type")}')
print(f'Balance:      ${p.get("balance", 0):,.0f}')
print(f'Credit Score: {p.get("credit_score")}')
print(f'Risk Level:   {p.get("risk_level")}')
print()
print(f'Trust Score:  {ctx["trust_score"]["score"]}/100 ({ctx["trust_score"]["tier"]})')
print(f'Transactions: {len(ctx["last_5_transactions"])} recent')
print(f'Sentiment:    {ctx["sentiment_trend"].get("latest_score", "N/A")}')
print()
if ctx.get('risk_profile'):
    rp = ctx['risk_profile']
    print(f'Risk Profile:')
    for k, v in rp.items():
        print(f'  {k}: {v}')
if ctx.get('recommendations'):
    print(f'\nTop Recommendations:')
    for r in ctx['recommendations'][:3]:
        print(f'  - {r.get("product", "?")}: {r.get("reason", "?")}')      